Load data from PostgreSQL

In [1]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.database import get_connection

query = "SELECT * FROM customers"

with get_connection() as connection:
    df = pd.read_sql(query, connection)

print("Shape:", df.shape)

Shape: (7043, 21)


C:\Users\Isha\AppData\Local\Temp\ipykernel_15088\306380575.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


In [2]:
df["churn"] = df["churn"].map({
    "Yes": 1,
    "No": 0
})

In [3]:
df["churn"].value_counts()

churn
0    5174
1    1869
Name: count, dtype: int64

In [4]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-1 Year",
        "1-2 Years",
        "2-4 Years",
        "4+ Years"
    ]
)

In [5]:
charge_threshold = df["monthly_charges"].median()

df["high_monthly_charge"] = (
    df["monthly_charges"] >= charge_threshold
).astype(int)

In [6]:
service_cols = [
    "phone_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

In [7]:
df["service_count"] = 0

df["service_count"] += (df["phone_service"] == "Yes").astype(int)

for col in service_cols[1:]:
    df["service_count"] += (df[col] == "Yes").astype(int)

In [8]:
df["service_count"].value_counts().sort_index()

service_count
0      80
1    2253
2     996
3    1041
4    1062
5     827
6     525
7     259
Name: count, dtype: int64

In [9]:
X = df.drop(
    columns=["customer_id", "churn"]
)

y = df["churn"]

In [10]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['senior_citizen', 'tenure', 'monthly_charges', 'total_charges', 'high_monthly_charge', 'service_count']

Categorical features:
['gender', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'tenure_group']


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [12]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [15]:
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (5634, 22)
Testing: (1409, 22)


In [19]:
print(df.columns.tolist())

['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn', 'tenure_group', 'high_monthly_charge', 'service_count']


In [20]:
print(df.shape)

(7043, 24)


In [21]:
df["churn"].value_counts()

churn
0    5174
1    1869
Name: count, dtype: int64

In [22]:
df["churn"].value_counts(normalize=True).mul(100).round(2)

churn
0    73.46
1    26.54
Name: proportion, dtype: float64

In [24]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

print("Training:", X_train_fe.shape)
print("Testing :", X_test_fe.shape)

Training: (5634, 22)
Testing : (1409, 22)


In [25]:
if "customer_id" in X_train_fe.columns:
    X_train_fe = X_train_fe.drop(columns=["customer_id"])
    X_test_fe = X_test_fe.drop(columns=["customer_id"])

print(X_train_fe.shape)

(5634, 22)


In [26]:
import numpy as np

for data in [X_train_fe, X_test_fe]:
    data["avg_monthly_spend"] = np.where(
        data["tenure"] > 0,
        data["total_charges"] / data["tenure"],
        data["monthly_charges"]
    )

In [27]:
bins = [-1, 12, 24, 48, 72]
labels = ["0-1 Year", "1-2 Years", "2-4 Years", "4+ Years"]

for data in [X_train_fe, X_test_fe]:
    data["tenure_group"] = pd.cut(
        data["tenure"],
        bins=bins,
        labels=labels
    )

In [28]:
service_columns = [
    "phone_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

service_columns = [
    col for col in service_columns
    if col in X_train_fe.columns
]

for data in [X_train_fe, X_test_fe]:
    data["total_services"] = (
        data[service_columns]
        .eq("Yes")
        .sum(axis=1)
    )

In [29]:
categorical_cols = X_train_fe.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X_train_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numerical_cols)

Categorical columns:
['gender', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'tenure_group']

Numerical columns:
['senior_citizen', 'tenure', 'monthly_charges', 'total_charges', 'high_monthly_charge', 'service_count', 'avg_monthly_spend', 'total_services']


C:\Users\Isha\AppData\Local\Temp\ipykernel_15088\3235263597.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train_fe.select_dtypes(


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols
        )
    ]
)

In [32]:
X_train_processed = preprocessor.fit_transform(X_train_fe)

X_test_processed = preprocessor.transform(X_test_fe)

print("Processed training:", X_train_processed.shape)
print("Processed testing :", X_test_processed.shape)

Processed training: (5634, 53)
Processed testing : (1409, 53)


In [33]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))
print(feature_names)

Number of features: 53
['num__senior_citizen' 'num__tenure' 'num__monthly_charges'
 'num__total_charges' 'num__high_monthly_charge' 'num__service_count'
 'num__avg_monthly_spend' 'num__total_services' 'cat__gender_Female'
 'cat__gender_Male' 'cat__partner_No' 'cat__partner_Yes'
 'cat__dependents_No' 'cat__dependents_Yes' 'cat__phone_service_No'
 'cat__phone_service_Yes' 'cat__multiple_lines_No'
 'cat__multiple_lines_No phone service' 'cat__multiple_lines_Yes'
 'cat__internet_service_DSL' 'cat__internet_service_Fiber optic'
 'cat__internet_service_No' 'cat__online_security_No'
 'cat__online_security_No internet service' 'cat__online_security_Yes'
 'cat__online_backup_No' 'cat__online_backup_No internet service'
 'cat__online_backup_Yes' 'cat__device_protection_No'
 'cat__device_protection_No internet service' 'cat__device_protection_Yes'
 'cat__tech_support_No' 'cat__tech_support_No internet service'
 'cat__tech_support_Yes' 'cat__streaming_tv_No'
 'cat__streaming_tv_No internet service

In [34]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_fe.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test_fe.index
)

print(X_train_processed.shape)
print(X_test_processed.shape)

(5634, 53)
(1409, 53)


In [35]:
print("Missing values in training:")
print(X_train_processed.isnull().sum().sum())

print("\nMissing values in testing:")
print(X_test_processed.isnull().sum().sum())

Missing values in training:
0

Missing values in testing:
0


In [36]:
print("Final X_train shape:", X_train_processed.shape)
print("Final X_test shape :", X_test_processed.shape)

print("\nFirst 5 rows:")
display(X_train_processed.head())

Final X_train shape: (5634, 53)
Final X_test shape : (1409, 53)

First 5 rows:


,num__senior_citizen,num__tenure,num__monthly_charges,num__total_charges,num__high_monthly_charge,num__service_count,num__avg_monthly_spend,num__total_services,cat__gender_Female,cat__gender_Male,...,cat__paperless_billing_No,cat__paperless_billing_Yes,cat__payment_method_Bank transfer (automatic),cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__tenure_group_0-1 Year,cat__tenure_group_1-2 Years,cat__tenure_group_2-4 Years,cat__tenure_group_4+ Years
3738,2.266551,1.614756,1.442539,2.422507,1.00071,1.660982,1.433873,1.660982,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3151,-0.441199,0.598676,0.355508,0.558933,1.00071,-0.511754,0.353243,-0.511754,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4862,-0.441199,-0.498691,-1.327477,-0.780331,-0.99929,-1.598122,-1.311274,-1.598122,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3868,-0.441199,-0.620621,1.257761,-0.235420,1.00071,1.117798,1.252150,1.117798,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3810,2.266551,-0.579977,0.292250,-0.404219,1.00071,-0.511754,0.356475,-0.511754,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [37]:
print(y_train.value_counts())
print()
print(y_test.value_counts())

churn
0    4139
1    1495
Name: count, dtype: int64

churn
0    1035
1     374
Name: count, dtype: int64


In [38]:
X_train_processed.to_csv(
    "../data/processed/X_train_processed.csv",
    index=False
)

X_test_processed.to_csv(
    "../data/processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
